# Static Machine Baseline

Train a simple baseline that uses only machine-level static features, ignoring the ONNX model itself.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
base_dir = next(
    candidate for candidate in [
        Path.cwd() / "training" / "data",
        Path.cwd().parent / "data",
    ]
    if (candidate / "train_set.csv").exists()
)

train_df = pd.read_csv(base_dir / "train_set.csv")
val_df = pd.read_csv(base_dir / "val_set.csv")
test_df = pd.read_csv(base_dir / "test_set.csv")

feature_cols = [
    "num_cores",
    "memory_mb",
    "l1d_cache_kb",
    "l1i_cache_kb",
    "l2_cache_kb",
    "base_clock_mhz",
    "memory_bandwith_gbs",
    "cpu_provider",
    "machine_type",
    "platform",
]
target_col = "average_ms"

numeric_features = [
    "num_cores",
    "memory_mb",
    "l1d_cache_kb",
    "l1i_cache_kb",
    "l2_cache_kb",
    "base_clock_mhz",
    "memory_bandwith_gbs",
]
categorical_features = ["cpu_provider", "machine_type", "platform"]

preprocess = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    [
        ("preprocess", preprocess),
        ("regressor", Ridge(alpha=1.0)),
    ]
)

X_train = train_df[feature_cols]
y_train = np.log(train_df[target_col].clip(lower=1e-9))
X_val = val_df[feature_cols]
y_val = np.log(val_df[target_col].clip(lower=1e-9))
X_test = test_df[feature_cols]
y_test = np.log(test_df[target_col].clip(lower=1e-9))

_ = model.fit(X_train, y_train)

In [13]:
def latency_metrics_from_log(y_log_true, y_log_pred):
    true_ms = np.exp(y_log_true)
    pred_ms = np.exp(y_log_pred)

    error_ms = pred_ms - true_ms
    rel_err = np.abs(error_ms) / true_ms
    ratio_err = np.maximum(pred_ms / true_ms, true_ms / pred_ms)

    rmse_log = np.sqrt(mean_squared_error(y_log_true, y_log_pred))
    rmse_ms = np.sqrt(np.mean(error_ms ** 2))
    rmse_percent = np.sqrt(np.mean(((error_ms / true_ms) * 100) ** 2))

    return {
        "rmse_log": rmse_log,
        "rmse_ms": rmse_ms,
        "rmse_percent": rmse_percent,
        "median_relative_error": np.median(rel_err),
        "p90_relative_error": np.percentile(rel_err, 90),
        "p95_relative_error": np.percentile(rel_err, 95),
        "median_percent_error": np.median(rel_err) * 100,
        "p90_percent_error": np.percentile(rel_err, 90) * 100,
        "p95_percent_error": np.percentile(rel_err, 95) * 100,
        "median_ratio_error": np.median(ratio_err),
        "p90_ratio_error": np.percentile(ratio_err, 90),
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
        "within_2x": np.mean(ratio_err <= 2.0),
    }


def evaluate(split_name, features, target_log):
    prediction_log = model.predict(features)
    metrics = latency_metrics_from_log(target_log, prediction_log)
    print(f"{split_name}:")
    for key, value in metrics.items():
        print(f"  {key}: {value:.4f}")


evaluate("train", X_train, y_train)
evaluate("validation", X_val, y_val)
evaluate("test", X_test, y_test)

train:
  rmse_log: 1.3956
  rmse_ms: 605.9120
  rmse_percent: 5771.4148
  median_relative_error: 0.6738
  p90_relative_error: 4.4590
  p95_relative_error: 8.9242
  median_percent_error: 67.3834
  p90_percent_error: 445.8988
  p95_percent_error: 892.4167
  median_ratio_error: 2.1589
  p90_ratio_error: 10.4145
  within_10pct: 0.0661
  within_25pct: 0.1762
  within_50pct: 0.3652
  within_2x: 0.4606
validation:
  rmse_log: 1.3976
  rmse_ms: 599.7018
  rmse_percent: 6029.9835
  median_relative_error: 0.6735
  p90_relative_error: 4.3771
  p95_relative_error: 8.3626
  median_percent_error: 67.3525
  p90_percent_error: 437.7132
  p95_percent_error: 836.2632
  median_ratio_error: 2.1620
  p90_ratio_error: 10.1529
  within_10pct: 0.0664
  within_25pct: 0.1776
  within_50pct: 0.3667
  within_2x: 0.4606
test:
  rmse_log: 1.3662
  rmse_ms: 582.7244
  rmse_percent: 5095.3424
  median_relative_error: 0.6690
  p90_relative_error: 4.2958
  p95_relative_error: 8.1308
  median_percent_error: 66.9006
  p9